<a href="https://colab.research.google.com/github/dongxiat/MyColabHub/blob/main/InvSR_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%cd /content
!rm -rf InvSR
!git clone https://huggingface.co/spaces/OAOA/InvSR
%cd /content/InvSR

# Loại bỏ scikit-image khỏi requirements.txt để dùng bản cài sẵn cực kỳ ổn định của Colab
!sed -i '/scikit-image/d' requirements.txt

# Cài đặt các thư viện còn lại
!pip install --upgrade pip -q
!pip install -r requirements.txt -q spaces loguru grad-cam

# Tạo file app.py mới hoàn toàn với thiết kế giao diện đẹp mắt và hỗ trợ upload nhiều ảnh
with open('/content/InvSR/app.py', 'w') as f:
    f.write('''import os
import gradio as gr
import torch
from sampler_invsr import InvSamplerSR
import numpy as np
from PIL import Image

# Khởi tạo mô hình
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

# Khởi tạo model sampler
model = InvSamplerSR(device=device)

def process_multiple_images(images, steps):
    if not images:
        return [], "Vui lòng tải lên ít nhất một hình ảnh!"

    output_images = []
    num_images = len(images)

    for i, img_path in enumerate(images):
        try:
            # Đọc ảnh
            img = Image.open(img_path).convert("RGB")
            # Thực hiện siêu phân giải với số bước steps
            sr_img = model.super_resolve(img, steps=int(steps))
            output_images.append(sr_img)
        except Exception as e:
            print(f"Lỗi xử lý ảnh thứ {i+1}: {str(e)}")

    return output_images, f"Đã xử lý thành công {len(output_images)}/{num_images} ảnh!"

# Thiết kế giao diện Gradio cao cấp
css = """
#col-container {max-width: 1050px; margin-left: auto; margin-right: auto;}
.animate-spin {animation: spin 1s linear infinite;}
@keyframes spin {from {transform: rotate(0deg);} to {transform: rotate(360deg);}}
#gallery-output {height: auto; min-height: 400px;}
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="cyan"), css=css) as demo:
    with gr.Column(elem_id="col-container"):
        gr.HTML("""
        <div style='text-align: center; margin-bottom: 20px;'>
            <h1 style='color: #0d9488; font-size: 2.5rem; font-weight: 800; margin-bottom: 10px;'>
                ✨ InvSR Super-Resolution Pro
            </h1>
            <p style='color: #4b5563; font-size: 1.1rem;'>
                Hệ thống siêu phân giải ảnh tiên tiến, phục hồi chi tiết sắc nét và xử lý hàng loạt ảnh chất lượng cao.
            </p>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=5):
                with gr.Group():
                    gr.Markdown("### 📥 Bước 1: Tải lên danh sách ảnh cần làm nét")
                    input_files = gr.File(
                        file_count="multiple",
                        file_types=["image"],
                        label="Chọn hoặc kéo thả nhiều file ảnh tại đây"
                    )

                with gr.Group(style="margin-top: 15px;"):
                    gr.Markdown("### ⚙️ Bước 2: Cấu hình thông số")
                    steps_slider = gr.Slider(
                        minimum=10,
                        maximum=200,
                        value=50,
                        step=10,
                        label="Số bước lặp (Steps) - Càng cao ảnh càng nét nhưng xử lý lâu hơn"
                    )

                submit_btn = gr.Button("🚀 Bắt đầu khôi phục ảnh", variant="primary", size="lg", style="margin-top: 15px;")

            with gr.Column(scale=5):
                with gr.Group():
                    gr.Markdown("### 🖼️ Kết quả khôi phục (Gallery)")
                    output_gallery = gr.Gallery(
                        label="Ảnh sau khi xử lý siêu phân giải",
                        elem_id="gallery-output",
                        columns=[2],
                        rows=[2],
                        object_fit="contain",
                        height="auto"
                    )
                    status_text = gr.Textbox(label="Trạng thái hệ thống", interactive=False, value="Đang chờ hình ảnh...")

        gr.HTML("""
        <div style='border-top: 1px solid #e5e7eb; padding-top: 20px; margin-top: 30px; text-align: center; color: #6b7280; font-size: 0.9rem;'>
            Được phát triển dựa trên cấu trúc InvSR Model • Hỗ trợ xử lý tăng tốc bằng GPU CUDA
        </div>
        """)

        submit_btn.click(
            fn=process_multiple_images,
            inputs=[input_files, steps_slider],
            outputs=[output_gallery, status_text]
        )

demo.queue().launch(share=True, debug=True)
''')


/content
Cloning into 'InvSR'...
remote: Enumerating objects: 1007, done.
remote: Total 1007 (delta 0), reused 0 (delta 0), pack-reused 1007 (from 1)
Receiving objects: 100% (1007/1007), 14.16 MiB | 27.25 MiB/s, done.
Resolving deltas: 100% (310/310), done.
/content/InvSR
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
import torch
torch.cuda.empty_cache()
%cd /content/InvSR
# Chạy trực tiếp bằng python thông thường thay vì uv run
!python app.py

/content/InvSR
Traceback (most recent call last):
  File "/content/InvSR/app.py", line 15, in <module>
    model = InvSamplerSR(device=device)
TypeError: BaseSampler.__init__() got an unexpected keyword argument 'device'
